# Lesson 5: Image Moments

Once we can isolate a blob (Lesson 4), moments let us summarize its shape with a handful of numbers: its area, centroid, orientation, and even a description that stays the same under translation, scale, and rotation. Moments offer a classic, lightweight alternative to learned features (covered later) for simple shape matching.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## What is a moment?

For a binary image, the raw moment $m_{pq}$ is defined as

$$m_{pq} = \sum_{x,y} x^p y^q \, I(x, y)$$

where $I(x,y)$ is 1 inside the shape and 0 elsewhere. A few special cases are already familiar quantities:

- $m_{00}$ = area (pixel count)
- ($\bar{x}, \bar{y}) = (m_{10}/m_{00}, m_{01}/m_{00})$ yields the centroid

The **order** of a moment is $p+q$. The zeroth-order moment is $m_{00}$; the first-order moments are $m_{10}$ and $m_{01}$; and the second-order moments are $m_{20}$, $m_{02}$, and $m_{11}$. `cv2.moments` computes all of these in one call, along with the *central* moments $\mu_{ij}$, which are translation-invariant; and the *Hu moments*, which are translation-, scale-, and rotation-invariant.

## Raw moments, computed by hand

The formula above is just a weighted sum over pixel coordinates, so for a binary image it's nothing more than: find every foreground pixel's $(x,y)$ coordinates, then sum $x^py^q$ over them. Here it is on a tiny 3x3 image with 3 foreground pixels, computed directly with NumPy and cross-checked against `cv2.moments`.

In [ ]:
im = np.array([
    [0, 1, 0],
    [1, 1, 0],
    [0, 0, 0],
], dtype=np.uint8)

ys, xs = np.nonzero(im)       # (row, col) = (y, x) of every foreground pixel

M00 = len(xs)                 # area: just a count of foreground pixels
M10 = xs.sum()                # sum of x^1 y^0 over foreground pixels
M01 = ys.sum()                # sum of x^0 y^1 over foreground pixels

m = cv2.moments(im, binaryImage=True)

print(f'foreground pixels (x, y): {[(int(x), int(y)) for x, y in zip(xs, ys)]}')
print(f'moments:     m00 = {M00}   m10 = {M10}   m01 = {M01}')
print(f"cv2.moments: m00 = {m['m00']:.0f}  m10 = {m['m10']:.0f}  m01 = {m['m01']:.0f}")
print(f'centroid = ({M10 / M00:.2f}, {M01 / M00:.2f})')

## An elongated, rotated blob

We draw a rotated ellipse so its orientation is easy to eyeball and check against what the moments compute.

In [ ]:
binary = np.zeros((200, 200), dtype=np.uint8)
center = (100, 100)
axes_len = (70, 25)
angle_deg = 30
cv2.ellipse(binary, center, axes_len, angle_deg, 0, 360, 255, -1)

plt.imshow(binary, cmap='gray')
plt.title(f'Ellipse drawn at {angle_deg} degrees')
plt.axis('off')
plt.show()

## Area and centroid from moments

In [ ]:
m = cv2.moments(binary, binaryImage=True)

area = m['m00']
cx = m['m10'] / m['m00']
cy = m['m01'] / m['m00']

print(f'area (m00)   = {area:.0f} pixels')
print(f'centroid     = ({cx:.1f}, {cy:.1f})')

plt.imshow(binary, cmap='gray')
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Centroid from moments')
plt.axis('off')
plt.show()

## Central moments

Raw moments $m_{pq}$ depend on where the shape happens to sit in the image &mdash; move the shape and every $m_{ij}$ (except $m_{00}$) changes. **Central moments** fix this by measuring around the shape's own centroid $(\bar x, \bar y)$ instead of the image origin:

$$\mu_{pq} = \sum_{x,y} (x-\bar x)^p\,(y-\bar y)^q\, I(x,y)$$

Recomputing this sum from scratch would mean re-visiting every pixel again. Instead, there's a shift-of-origin identity &mdash; the image-moment analogue of $\mathrm{Var}(X) = E[X^2] - E[X]^2$ &mdash; that computes the central moments directly from the raw moments:

$$\mu_{20} = M_{20} - \bar x\,m_{10}, \qquad \mu_{02} = m_{02} - \bar y\,m_{01}, \qquad \mu_{11} = m_{11} - \bar x\,m_{01}$$

`cv2.moments` returns both raw (`m['m20']`, ...) and central (`m['mu20']`, ...) moments in the same dictionary, so in practice these are never computed by hand &mdash; but now you know where `mu20`, `mu11`, and `mu02` actually come from.

In [ ]:
mu20_manual = m['m20'] - cx * m['m10']
mu02_manual = m['m02'] - cy * m['m01']
mu11_manual = m['m11'] - cx * m['m01']

print(f'moments:     mu20 = {mu20_manual:.1f}   mu02 = {mu02_manual:.1f}   mu11 = {mu11_manual:.1f}')
print(f"cv2.moments: mu20 = {m['mu20']:.1f}   mu02 = {m['mu02']:.1f}   mu11 = {m['mu11']:.1f}")

## Orientation from central moments

The central moments $\mu_{20}$, $\mu_{02}$, $\mu_{11}$ describe the spread of the shape around its centroid — essentially its covariance matrix (which we revisit in Lesson 6). The angle of the major axis (the direction of greatest spread) is

$$\theta = \frac{1}{2}\,\mathrm{atan2}\!\left(2\mu_{11},\; \mu_{20} - \mu_{02}\right)$$

In [ ]:
theta = 0.5 * np.arctan2(2 * m['mu11'], m['mu20'] - m['mu02'])
theta_deg = np.degrees(theta)

print(f'orientation from moments = {theta_deg:.1f} degrees')
print(f'angle used to draw the ellipse = {angle_deg} degrees')

# The eigenvalues of the (normalized) covariance matrix of central moments
# give the axis lengths of the equivalent ellipse: axis = 4*sqrt(eigenvalue).
cov = np.array([[m['mu20'], m['mu11']], [m['mu11'], m['mu02']]]) / m['m00']
eigvals, _ = np.linalg.eigh(cov)
semi_major = 2 * np.sqrt(eigvals[-1])

dx, dy = semi_major * np.cos(theta), semi_major * np.sin(theta)

plt.imshow(binary, cmap='gray')
plt.plot([cx - dx, cx + dx], [cy - dy, cy + dy], c='red', linewidth=2)
plt.scatter(cx, cy, c='red', marker='x', s=80)
plt.title('Major axis recovered from moments')
plt.axis('off')
plt.axis('equal')
plt.show()

## Hu moments: a shape descriptor invariant to pose

**Hu moments** (`cv2.HuMoments`) are 7 values calculated from the central moments that stay (nearly) the same regardless of the shape's position, size, and rotation. This makes them useful for comparing two shapes without first aligning them.

To see this, we build three versions of a dog shape: at the original pose, translated, and rotated + scaled. We compare the Hu moments of these versions of the shape with each other, and with a different (cat) shape.

In [ ]:
def warp(im, mat):
    rows, cols = im.shape[:2]
    return cv2.warpAffine(im, np.float32(mat), (cols, rows))

im_original = cv2.imread('../img/dog_clipart.png', cv2.IMREAD_GRAYSCALE)
im_translated = warp(im_original, [[1, 0, 10], [0, 1, 20]])
im_rotated_scaled = warp(im_original, cv2.getRotationMatrix2D([60, 60], angle=45, scale=0.5))
im_different_shape = cv2.imread('../img/cat_clipart.png', cv2.IMREAD_GRAYSCALE)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img, title in zip(
    axes,
    [im_original, im_translated, im_rotated_scaled, im_different_shape],
    ['Original', 'Translated', 'Rotated + scaled', 'Different shape'],
):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p style="font-size: 0.8em; color: #888; text-align: right;">Image source: <a href="https://www.publicdomainpictures.net" target="_blank" rel="noopener">Public domain pictures</a></p>

Now let's compute the Hu moments.

In [ ]:
def hu_log(im):
    m = cv2.moments(im, binaryImage=True)
    hu = cv2.HuMoments(m).flatten()
    # log-scale since raw Hu moments span many orders of magnitude
    return -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)

# Format with a fixed width (":8.3f") to align the values across rows.
label_width, col_width, n_hu = 18, 8, 7
print(' ' * (label_width + 2) + 'Hu moments'.center(col_width * n_hu))
print(f'{"image":>{label_width}}  ' + ''.join(f'{i:>{col_width}}' for i in range(1, n_hu + 1)))
for im, name in [
    (im_original, 'original'),
    (im_translated, 'translated'),
    (im_rotated_scaled, 'rotated+scaled'),
    (im_different_shape, 'different shape'),
]:
    row = ''.join(f'{v:{col_width}.3f}' for v in hu_log(im))
    print(f'{name:>{label_width}}  {row}')

Translated and rotated+scaled versions yield similar Hu moments, whereas the different shape yields much different values. This is exactly the invariance property that makes Hu moments useful for shape matching.

### Exercise

1. Use `cv2.findContours` to get the outline of a blob from Lesson 4's binary image, then call `cv2.moments` on the *contour* instead of the full binary mask. Compare the centroid to the one computed here.
2. Draw a shape that is mirror-flipped rather than rotated. Are its Hu moments still close to the original? (Hint: think about what determinant/parity information moments do or do not capture.)